In [1]:
# %%
import jax
import jax.numpy as jnp
import numpy as onp
%load_ext autoreload
%autoreload 2
import jax_sn

jax.config.update("jax_enable_x64", True)
import equinox as eqx
import warp as wp 
from jax_sn.tools.setup_test_mgop import setup_slab_mgop, make_xs_test, setup_layered_slab_mgop
from jax_sn.operators.transport_operator import TransportOperatorBatchedPallas


# %%
N_PER_DIM = 20

def _build_base_mgop():
    """Build the reference mgop with default (Vmap) operators. Expensive — call once."""
    xs = [make_xs_test(m, n_groups=2, l_order=2) for m in ["fuel", "moderator", "reflector"]]
    return setup_layered_slab_mgop(
        dim=3,
        degree=1,
        tn_quadrature_set=3,
        region_sizes=[1.0, 4.0, 2.0],
        region_resolutions=[N_PER_DIM - 3, 2, 1],
        region_xs=xs,
        region_sources=[1.0, 0.5, 0.2],
        transverse_specs=[(2.0, N_PER_DIM), (1.523, N_PER_DIM)],
        use_numpy=True,
        single_angle_transport_operator_cls = TransportOperatorBatchedPallas, single_angle_transport_operator_cls_kwargs = {"batch_size": 100}
    )

onp.random.seed(2042026)


base_mgop = _build_base_mgop()
b_apply = jnp.array(onp.random.random_sample(base_mgop.transport_operator.out_structure().shape))

In [2]:
# %%
from jax_sn.operators.transport_operator.transport_operator_warp import TransportOperatorSequentialWarpSingleAngle, TransportOperatorSequentialWarpMultipleAngles, TransportOperatorLevelsWarpSingleAngle, TransportOperatorLevelsWarpMultipleAngles
top_sa_levels = TransportOperatorLevelsWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator.topological_sorted_angle[0],0, batch_size=100, max_n_levels= None)
top_ma_levels = TransportOperatorLevelsWarpMultipleAngles(TransportOperatorLevelsWarpSingleAngle(base_mgop.solution_domain_data, base_mgop.transport_operator.topological_sorted_angle, 0, batch_size = 100, max_n_levels=None))



In [3]:
from jax_sn.operators.multi_group_operator.lower_triangular_solver import MultiGroupLowerTriangularBlockSolver, SolverDiagnostics
from jax_sn.operators.multi_group_operator.right_hand_side import IsotropicSourceSingleGroup, solve_multigroup_operator_rhs
import lineax as lx
jax.clear_caches()

solver = MultiGroupLowerTriangularBlockSolver(lx.BiCGStab(1e-10, 1e-10, max_steps = 200), SolverDiagnostics(print_single_group = True, print_sweep = False))

rhs = IsotropicSourceSingleGroup.from_operator(base_mgop, jnp.ones((base_mgop.n_elements, base_mgop.n_basis)))

In [4]:
solution_0 = solve_multigroup_operator_rhs(base_mgop, rhs, solver )

Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps


In [5]:
mgop_levels = eqx.tree_at(lambda op: op.transport_operator, base_mgop, top_ma_levels)

In [6]:
solution_1 = solve_multigroup_operator_rhs(mgop_levels, rhs, solver )

Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.1
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3080" (10 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/tbogaarts/.cache/warp/1.12.0
Module jax_sn.operators.transport_operator.linear_tetrahedron_warp_kernel 03b08e1 load on device 'cuda:0' took 4.94 ms  (cached)
Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps


In [7]:
%timeit solve_multigroup_operator_rhs(mgop_levels, rhs, solver ).value.block_until_ready()

Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
Using lower triangular transpose solver for multi-group operator with 2 energy groups.
Solved energy group 0 in 6 steps
Solved energy group 1 in 7 steps
Using lower triangular transpose solver for multi-group operator with 2 energy gro

In [8]:
sgop_base = base_mgop.to_single_group_operator(0)

rhs_2 = jnp.ones_like(sgop_base.out_structure())

In [9]:
%timeit sgop_base.mv(rhs_2).block_until_ready()

157 ms ± 1.69 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
sgop_2 = mgop_levels.to_single_group_operator(0)

In [11]:
%timeit sgop_2.mv(rhs_2).block_until_ready()

27.1 ms ± 349 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [12]:
rhs_3 = jnp.ones_like(sgop_base.moment_operator.in_structure())

rhs_4 = jnp.ones_like(sgop_base.discrete_operator.in_structure())

rhs_5 = jnp.ones_like(sgop_base.mass_operator.in_structure())

In [13]:
%timeit sgop_2.mass_operator.mv(rhs_4).block_until_ready()

1.57 ms ± 33.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
%timeit -n 40 -r 3 top_ma_levels.inv(b_apply).block_until_ready()

16.6 ms ± 2.06 ms per loop (mean ± std. dev. of 3 runs, 40 loops each)


In [15]:
%timeit -n 40 -r 3 top_ma_levels.inv(b_apply).block_until_ready()

15 ms ± 138 μs per loop (mean ± std. dev. of 3 runs, 40 loops each)


In [16]:
%timeit  base_mgop.transport_operator.inv(b_apply).block_until_ready()

144 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
def _call_batched_moment_operator_transposed(x : jnp.ndarray, wk_yklm : jnp.ndarray) -> jnp.ndarray:
    r'''
    Apply the transposed moment operator to the angles to compute the angular flux buffer.

    Computes:   

    .. math::
        \psi_k^{ei} = \sum_{l,m} x_{l,m}^{ei} w_{k} Y_{l,m}(\Omega_k)    

    with :math:`\psi_k^{ei}` the angular flux for angle :math:`k` in element :math:`e` and basis function :math:`i`, :math:`x_{l,m}^{ei}` the moment of order :math:`(l,m)` in element :math:`e` and basis function :math:`i`,
    :math:`w_k` the weight for angle :math:`k` and :math:`Y_{l,m}(\Omega_k)` the spherical harmonic of order :math:`(l,m)` evaluated at direction :math:`\Omega_k`. For the latter definition, 
    see `jax_sn.quadrature_set.QuadratureSet.wk_ylm_real`.  

    This does *not* invert the transformation performed by the moment operator, since the weights are included here as well.

    Parameters:
    ----------
    x : jnp.ndarray [(L+1)**2, n_elements, n_basis]
        The angular flux buffer containing the angular fluxes for each angle and each element and its basis functions
    wk_yklm : jnp.ndarray [n_angles, (L+1)**2]
        The moment operator weights mapping angles to moments for the specific desired order L.

    Returns:
    -------
    moments : jnp.ndarray [n_angles, n_elements, n_basis]
        The computed moments.
    
    '''    
    assert x.shape[0] == wk_yklm.shape[-1], f"Number of moments in x ({x.shape[0]}) must match number of moments in wk_yklm ({wk_yklm.shape[-1]})."
    return jnp.einsum('l...,kl->k...', x, wk_yklm)


In [46]:
aaa1 = sgop_2.mv(rhs_2).block_until_ready()

In [55]:
jax.clear_caches()

In [56]:
bb1 = sgop_2.mv(rhs_2).block_until_ready()

In [57]:
onp.testing.assert_allclose(bb1, aaa1)

In [59]:
from jax_sn.tools.memory_analysis import OperatorAnalysis
OperatorAnalysis.analyze_timing(sgop_2.mass_operator, "mv", n_samples = 100)

np.float64(0.012248650945798314)

In [18]:
import timeit 

timeit.timeit(lambda : sgop_2.mv(rhs_2).block_until_ready(), number = 2)

0.06082398899980035

In [43]:
timeit.time.time()

1775552643.8045611